# Домашнее задание №3

Есть данные по транзакциям клиентов, ваша задача состоит в анализе этих данных и подготовки к структуре, которая похожа на ту структуру, которая сейчас часто нами используется при построении моделей на транзакциях + промежуточные задания.

Не забудьте делать всякие `.show()` после каждого задания, чтобы было видно результат.

**Файл spark_transactions.parquet можете забрать в папке с записями лекций**

**Важно:** В домашнем задании старайтесь использовать максимально dataframe api, а не sql запросы.

In [1]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("homework")
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
    .getOrCreate()
)
sc = spark.sparkContext

In [3]:
trans_data = spark.read.parquet("spark_transactions.parquet")

In [4]:
trans_data.count()

24386900

In [5]:
trans_data.show(5)

+----+----+----+-----+---+-----+-------+-----------------+-------------+-------------+-------+----+-------+
|User|Card|Year|Month|Day| Time| Amount|          UseChip| MerchantCity|MerchantState|    Zip| MCC|IsFraud|
+----+----+----+-----+---+-----+-------+-----------------+-------------+-------------+-------+----+-------+
|   0|   0|2002|    9|  1|06:21|$134.09|Swipe Transaction|     La Verne|           CA|91750.0|5300|     No|
|   0|   0|2002|    9|  1|06:42| $38.48|Swipe Transaction|Monterey Park|           CA|91754.0|5411|     No|
|   0|   0|2002|    9|  2|06:22|$120.34|Swipe Transaction|Monterey Park|           CA|91754.0|5411|     No|
|   0|   0|2002|    9|  2|17:45|$128.95|Swipe Transaction|Monterey Park|           CA|91754.0|5651|     No|
|   0|   0|2002|    9|  3|06:23|$104.71|Swipe Transaction|     La Verne|           CA|91750.0|5912|     No|
+----+----+----+-----+---+-----+-------+-----------------+-------------+-------------+-------+----+-------+
only showing top 5 rows


Посмотрим на схему данных

In [6]:
trans_data.printSchema()

root
 |-- User: long (nullable = true)
 |-- Card: long (nullable = true)
 |-- Year: long (nullable = true)
 |-- Month: long (nullable = true)
 |-- Day: long (nullable = true)
 |-- Time: string (nullable = true)
 |-- Amount: string (nullable = true)
 |-- UseChip: string (nullable = true)
 |-- MerchantCity: string (nullable = true)
 |-- MerchantState: string (nullable = true)
 |-- Zip: double (nullable = true)
 |-- MCC: long (nullable = true)
 |-- IsFraud: string (nullable = true)



Сколько в среднем транзакций у пользователя

In [7]:
trans_data.groupby("User").count().agg(F.avg(F.col("count"))).collect()[0][0]

12193.45

Сколько карт у пользователей в среднем

In [8]:
trans_data.groupby("User").agg(F.count_distinct(F.col("Card")).alias("card_count")).agg(F.avg(F.col("card_count"))).collect()[0][0]

3.0695

Немного обработаем данные: Amount в float, из Time вытянем час транзакции и удалим исходный Time, Zip  к типу int

In [9]:
trans_data = (
    trans_data
    .withColumn("Amount", F.regexp_replace(F.col("Amount"), r"[\$,]", "").astype("float"))
    .withColumn("Hour", F.split(F.col("Time"), ":")[0].cast("int"))
    .drop("Time")
    .withColumn("Zip", F.col("Zip").astype("int"))
)

Посчитайте количество транзакций по годам, учитывая только те транзакции, объем которых был больше 100

In [10]:
trans_data.filter(F.col("Amount") > 100.0).groupby("Year").count().show()

+----+------+
|Year| count|
+----+------+
|2007|121489|
|2014|179492|
|2012|174439|
|1991|   266|
|2016|182742|
|1994|  1533|
|2018|182215|
|1999| 13656|
|1997|  5593|
|2009|150327|
|2010|165024|
|2006|105907|
|2017|182434|
|1998|  9009|
|2013|178456|
|2004| 69846|
|2003| 54288|
|2002| 40944|
|2011|172273|
|2020| 34925|
+----+------+
only showing top 20 rows


Определите, есть ли пропуски в данных по каждому столбцу

In [11]:
trans_data.select([F.count(F.when(F.isnan(F.col(c).try_cast("double")) | F.col(c).isNull(), c)).alias(c) for c in trans_data.columns]).show()

+----+----+----+-----+---+------+-------+------------+-------------+-------+---+-------+----+
|User|Card|Year|Month|Day|Amount|UseChip|MerchantCity|MerchantState|    Zip|MCC|IsFraud|Hour|
+----+----+----+-----+---+------+-------+------------+-------------+-------+---+-------+----+
|   0|   0|   0|    0|  0|     0|      0|           0|      2720821|2878135|  0|      0|   0|
+----+----+----+-----+---+------+-------+------------+-------------+-------+---+-------+----+



Заполните пропуски исходя из типа данных

In [12]:
trans_data = trans_data.fillna({"MerchantState": "UNKNOWN", "Zip": 0})

При помощи оконных функций для каждого клиента рассчитайте средний размер транзакции, количество транзакций и последнюю по дате транзакцию.

In [13]:
window = Window.partitionBy("User")
(
    trans_data
    .withColumn("Date", F.make_date("Year", "Month", "Day"))
    .withColumn("AvgAmount", F.avg(F.col("Amount")).over(window))
    .withColumn("Count", F.count("*").over(window))
    .withColumn("LastDate", F.max("Date").over(window))
    .select("User", "AvgAmount", "Count", "LastDate")
    .show()
)

+----+-----------------+-----+----------+
|User|        AvgAmount|Count|  LastDate|
+----+-----------------+-----+----------+
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
|   0|81.29998946268704|19963|2020-02-28|
+----+-----------------+-----+----

Теперь самое время сгруппировать данные по каждому клиенту (можно использовать `collect_list()` для сбора данных после агрегации).
Когда будете делать агрегацию, то возьмите только часть выборки, например, через `sample`, для всей выборки либо не хватит памяти, либо очень долго считать

In [14]:
grouped = (
    trans_data
    .sample(withReplacement=False, fraction=0.01, seed=42)
    .withColumn("Date", F.make_date("Year", "Month", "Day"))
    .groupBy("User", "Card")
    .agg(
        F.collect_list(
            F.struct("Date", "Amount", "Year", "Month", "Day", "Hour", "MCC")
        ).alias("transactions")
    )
)
grouped.show(1, truncate=False, vertical=True)

-RECORD 0----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
 User         | 0                                                                                                                                                                                                                                                                                                                           

Напишите python-функцию, которая возьмёт данные после агрегации последовательностей, отсортирует их внутри по дате и времени и преобразует к формату python `dict:
{'User': User,
'Card': Card,
'sequence':{
    'amount': [последовательность],
    'year': [последовательность],
    'month': [последовательность],
    'day': [последовательность],
    'time': [последовательность],
    'MCC': [последовательность]
}
}
`



In [15]:
def process_user_group(row):
    sorted_txs = sorted(row["transactions"], key=lambda x: x["Date"] if x["Date"] else "")

    return {
        "User": row["User"],
        "Card": row["Card"],
        "sequence": {
            "amount": [tx["Amount"] for tx in sorted_txs],
            "year": [tx["Year"] for tx in sorted_txs],
            "month": [tx["Month"] for tx in sorted_txs],
            "day": [tx["Day"] for tx in sorted_txs],
            "hour": [tx["Hour"] for tx in sorted_txs],
            "MCC": [tx["MCC"] for tx in sorted_txs],
        }
    }

final_data = grouped.rdd.map(process_user_group).collect()

Выведите как пример одну преобразованную запись, результаты сохраните на диск в через rdd pickle

In [16]:
final_data[0]

{'User': 0,
 'Card': 1,
 'sequence': {'amount': [10.970000267028809,
   31.889999389648438,
   143.24000549316406,
   50.0099983215332,
   87.58999633789062,
   32.630001068115234,
   72.0,
   44.36000061035156,
   223.5800018310547,
   30.239999771118164,
   93.45999908447266,
   62.34000015258789,
   144.36000061035156,
   86.01000213623047,
   -74.0],
  'year': [2014,
   2014,
   2014,
   2015,
   2015,
   2015,
   2015,
   2016,
   2016,
   2016,
   2016,
   2017,
   2019,
   2019,
   2019],
  'month': [5, 6, 12, 4, 8, 9, 9, 4, 8, 9, 10, 11, 3, 3, 6],
  'day': [16, 22, 30, 15, 4, 20, 22, 17, 26, 7, 26, 6, 5, 28, 9],
  'hour': [13, 6, 6, 6, 6, 6, 13, 6, 13, 5, 6, 9, 6, 20, 13],
  'MCC': [5541,
   5411,
   5541,
   5942,
   5912,
   5411,
   5541,
   5912,
   8021,
   5411,
   5912,
   7538,
   5300,
   5814,
   5499]}}